In [0]:
%sql
create or replace temporary view customers_raw_view as 
select * from (
    select distinct customer_id, name, coalesce(email, 'not provided') as email, city, state, signup_date, phone, ingestion_date
    from identifier(:catalog || '.bronze.customers_raw')
    where customer_id is not null

)
qualify row_number() over (partition by customer_id order by ingestion_date desc) = 1

In [0]:
%sql
create table if not exists identifier(:catalog || '.silver.customers_clean') (
    id int,
    name string,
    email string,
    city string,
    state string,
    signup_date date,
    phone string
)

In [0]:
%sql
merge into identifier(:catalog || '.silver.customers_clean') t
using customers_raw_view s
on t.id = s.customer_id
when matched then update set t.name = s.name, t.email = s.email, t.city = s.city, t.state = s.state, t.signup_date = s.signup_date, t.phone = s.phone
when not matched then insert (id, name, email, city, state, signup_date, phone) values (s.customer_id, s.name, s.email, s.city, s.state, s.signup_date, s.phone)